# ESG Company Screener

Upload one report per company, map each file to its ticker, and create an evidence scorecard and provisional ranking. The news scan combines GDELT with Google News RSS and uses FinBERT to flag negative English-language coverage for review.

The ranking is a provisional disclosure-evidence ranking adjusted only for analyst-approved incidents. It is not a verified ESG performance rating. News sentiment is a review signal, not proof of misconduct.


## Run sequence

1. Install the packages in `requirements_esg.txt` and restart the kernel. OCR also needs the Tesseract application installed on the computer.
2. Upload multiple PDFs. In the mapping box, enter one row per report as `filename.pdf | TICKER | Legal company name | alias1; alias2`. The last two fields are optional.
3. Run the cells in order. Review extracted report evidence and news candidates for each company.
4. Add confirmed incidents to the review table. Require five independent sources or a final regulator finding, court judgment or official sanction. Only approved incidents reduce scores.
5. Review the ranked scorecard and exported CSV files in `outputs`.

Google News RSS and GDELT are discovery sources. The prototype queries up to 90 days and records coverage; this is not a complete 12-month news archive. FinBERT classifies sentiment, not ESG violations.


In [ ]:
# Install dependencies in the active Jupyter environment, then restart the kernel.
from pathlib import Path
import subprocess, sys
requirements_file = next(path for path in (Path('requirements.txt'), Path('requirements_esg.txt'), Path('outputs/requirements_esg.txt')) if path.exists())
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_file)])


## 1. Upload the report and identify the company

In [ ]:
from pathlib import Path
from datetime import date, datetime, timedelta
from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode
from difflib import SequenceMatcher
from email.utils import parsedate_to_datetime
import json
import re
import time
import warnings
import xml.etree.ElementTree as ET

import pandas as pd
import requests
import pymupdf
import ipywidgets as widgets
from IPython.display import display, Markdown

MODEL_ID = "ProsusAI/finbert"
NEWS_LOOKBACK_DAYS = 365
NEWS_SOURCE_WINDOW_DAYS = 90
GDELT_MAX_RECORDS = 250
MAX_ARTICLE_FETCHES_PER_COMPANY = 75
NEGATIVE_PROBABILITY_REVIEW_FLAG = 0.70
OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pdf_upload = widgets.FileUpload(accept=".pdf", multiple=True, description="Upload PDFs")
mapping_input = widgets.Textarea(
    description="Company map",
    placeholder="report_a.pdf | ABC.NS | Example Company | Example; ABC",
    layout=widgets.Layout(width="95%", height="130px"),
)
display(widgets.VBox([pdf_upload, mapping_input]))

def uploaded_pdf_records(upload_widget):
    value = upload_widget.value
    if not value:
        raise ValueError("Upload at least one PDF before continuing.")
    if isinstance(value, dict):
        records = []
        for key, item in value.items():
            metadata = item.get("metadata", {})
            records.append({"name": metadata.get("name", key), "content": item.get("content")})
    else:
        records = [{"name": item.get("name", "uploaded_report.pdf"), "content": item.get("content")} for item in value]
    for item in records:
        if not str(item["name"]).lower().endswith(".pdf") or not item["content"]:
            raise ValueError(f"Invalid or empty PDF upload: {item['name']}")
        item["content"] = bytes(item["content"])
    return records


Upload the PDFs, then enter one mapping row per file using `filename.pdf | TICKER | Legal company name | alias1; alias2`. The ticker is required and must be unique. Then run the next cell.


In [ ]:
uploaded_files = uploaded_pdf_records(pdf_upload)
file_by_name = {item["name"]: item for item in uploaded_files}
company_inputs = []
for line_number, raw_line in enumerate(mapping_input.value.splitlines(), start=1):
    line = raw_line.strip()
    if not line:
        continue
    parts = [part.strip() for part in line.split("|")]
    if len(parts) < 2 or len(parts) > 4:
        raise ValueError(f"Mapping row {line_number} must have 2 to 4 fields separated by |.")
    filename, ticker = parts[0], parts[1].upper()
    if filename not in file_by_name:
        raise ValueError(f"Mapping row {line_number} refers to an uploaded filename not found: {filename}")
    company_name = parts[2] if len(parts) > 2 else ""
    aliases = [value.strip() for value in parts[3].split(";") if value.strip()] if len(parts) > 3 else []
    company_inputs.append({**file_by_name[filename], "ticker": ticker, "company_name": company_name, "aliases": aliases})
if not company_inputs:
    raise ValueError("Add a mapping row for every uploaded PDF.")
if len({item["ticker"] for item in company_inputs}) != len(company_inputs):
    raise ValueError("Each ticker must be unique in this comparison. Use one report per ticker.")
if set(file_by_name) != {item["name"] for item in company_inputs}:
    missing = sorted(set(file_by_name) - {item["name"] for item in company_inputs})
    raise ValueError(f"Add mapping rows for every uploaded file: {missing}")
print(pd.DataFrame([{ "file": c["name"], "ticker": c["ticker"], "company": c["company_name"] or c["ticker"] } for c in company_inputs]).to_string(index=False))
news_scan_date = date.today()


## 2. Load the user supplied topic dictionaries

The Social terms identify CSR, community investment and social impact evidence. They mostly describe activities and claims, so the notebook reports topic coverage rather than counting mentions as positive outcomes. Environmental and Governance categories use the supplied taxonomy excerpts. Review the embedded dictionaries before relying on scores because the pasted files were truncated.

In [ ]:
TAXONOMY = {
  "environment_sectors": [
    "Climate Change & GHG Emissions",
    "Carbon Management & Net Zero",
    "Energy Management",
    "Renewable Energy",
    "Air Quality & Pollution",
    "Water & Wastewater",
    "Waste Management",
    "Circular Economy",
    "Biodiversity & Ecosystems",
    "Land Use & Deforestation",
    "Natural Resources & Materials",
    "Chemicals & Hazardous Materials",
    "Environmental Compliance",
    "Environmental Management",
    "Climate Risk & Resilience",
    "Green Buildings & Infrastructure",
    "Sustainable Transportation",
    "Clean Technology",
    "Sustainable Agriculture & Food",
    "Oceans & Marine Environment",
    "Carbon Markets & Climate Finance",
    "Product Lifecycle & Eco-design",
    "Sustainable Supply Chain",
    "Environmental Reporting & Metrics",
    "Environmental Remediation"
  ],
  "environment_keywords": {
    "Climate Change & GHG Emissions": [
      "climate change",
      "global warming",
      "climate crisis",
      "climate impact",
      "climate action",
      "climate mitigation",
      "climate adaptation",
      "climate transition",
      "greenhouse gas",
      "greenhouse gases",
      "ghg",
      "ghg emissions",
      "carbon emissions",
      "carbon dioxide",
      "co2",
      "co2 emissions",
      "carbon dioxide equivalent",
      "co2e",
      "methane",
      "ch4",
      "methane emissions",
      "nitrous oxide",
      "n2o",
      "hydrofluorocarbons",
      "hfc",
      "hfcs",
      "perfluorocarbons",
      "pfc",
      "pfcs",
      "sulfur hexafluoride",
      "sf6",
      "nitrogen trifluoride",
      "nf3",
      "fugitive emissions",
      "process emissions",
      "direct emissions",
      "indirect emissions",
      "scope 1",
      "scope 1 emissions",
      "scope 2",
      "scope 2 emissions",
      "scope 3",
      "scope 3 emissions",
      "value chain emissions",
      "upstream emissions",
      "downstream emissions",
      "embodied carbon",
      "operational carbon",
      "emissions intensity",
      "carbon intensity",
      "emissions reduction",
      "emission reduction",
      "absolute emissions",
      "avoided emissions",
      "financed emissions",
      "temperature rise",
      "temperature alignment",
      "1.5 degree",
      "1.5°c",
      "2 degree scenario",
      "paris agreement",
      "global temperature",
      "climate target",
      "climate goal"
    ],
    "Carbon Management & Net Zero": [
      "net zero",
      "net-zero",
      "net zero emissions",
      "carbon neutral",
      "carbon neutrality",
      "climate neutral",
      "climate neutrality",
      "decarbonization",
      "decarbonisation",
      "decarbonize",
      "decarbonise",
      "deep decarbonization",
      "carbon reduction",
      "carbon abatement",
      "carbon mitigation",
      "carbon management",
      "carbon strategy",
      "carbon roadmap",
      "decarbonization roadmap",
      "carbon footprint",
      "corporate carbon footprint",
      "product carbon footprint",
      "carbon accounting",
      "ghg accounting",
      "carbon inventory",
      "ghg inventory",
      "carbon budget",
      "internal carbon price",
      "internal carbon pricing",
      "shadow carbon price",
      "carbon removal",
      "carbon capture",
      "carbon capture and storage",
      "ccs",
      "carbon capture utilization",
      "carbon capture utilisation",
      "ccus",
      "direct air capture",
      "dac",
      "carbon sequestration",
      "biological sequestration",
      "carbon sink",
      "nature based solutions",
      "nature-based solutions",
      "negative emissions",
      "science based target",
      "science-based target",
      "sbti",
      "emissions pathway",
      "transition pathway"
    ],
    "Energy Management": [
      "energy",
      "energy consumption",
      "energy use",
      "energy usage",
      "energy management",
      "energy efficiency",
      "energy efficient",
      "energy conservation",
      "energy savings",
      "energy reduction",
      "energy intensity",
      "energy productivity",
      "electricity consumption",
      "electricity use",
      "power consumption",
      "power usage",
      "grid electricity",
      "grid power",
      "energy mix",
      "fuel consumption",
      "fuel efficiency",
      "thermal energy",
      "heat recovery",
      "waste heat recovery",
      "combined heat and power",
      "chp",
      "cogeneration",
      "smart grid",
      "microgrid",
      "energy storage",
      "battery storage",
      "battery energy storage",
      "bess",
      "demand response",
      "energy audit",
      "energy management system",
      "iso 50001",
      "energy optimization",
      "energy optimisation",
      "energy performance",
      "energy transition",
      "electrification",
      "clean electricity"
    ],
    "Renewable Energy": [
      "renewable energy",
      "renewable power",
      "clean energy",
      "green energy",
      "low carbon energy",
      "solar",
      "solar energy",
      "solar power",
      "solar photovoltaic",
      "photovoltaic",
      "pv",
      "rooftop solar",
      "solar farm",
      "solar plant",
      "wind",
      "wind energy",
      "wind power",
      "wind farm",
      "offshore wind",
      "onshore wind",
      "hydropower",
      "hydroelectric",
      "hydroelectricity",
      "small hydro",
      "biomass",
      "bioenergy",
      "biogas",
      "renewable natural gas",
      "rng",
      "geothermal",
      "geothermal energy",
      "tidal energy",
      "wave energy",
      "green hydrogen",
      "renewable hydrogen",
      "hydrogen energy",
      "renewable electricity",
      "renewable electricity certificate",
      "renewable energy certificate",
      "rec",
      "recs",
      "renewable power purchase agreement",
      "power purchase agreement",
      "ppa",
      "clean power",
      "renewable capacity"
    ],
    "Air Quality & Pollution": [
      "air quality",
      "air pollution",
      "air pollutant",
      "air pollutants",
      "air emissions",
      "industrial emissions",
      "atmospheric emissions",
      "particulate matter",
      "pm10",
      "pm2.5",
      "pm 10",
      "pm 2.5",
      "nitrogen oxide",
      "nitrogen oxides",
      "nox",
      "nitrogen dioxide",
      "no2",
      "sulphur oxide",
      "sulfur oxide",
      "sox",
      "sulphur dioxide",
      "sulfur dioxide",
      "so2",
      "carbon monoxide",
      "co emissions",
      "volatile organic compounds",
      "volatile organic compound",
      "voc",
      "vocs",
      "ozone",
      "ground level ozone",
      "acid rain",
      "smog",
      "black carbon",
      "soot",
      "dust emissions",
      "stack emissions",
      "flue gas",
      "flaring",
      "gas flaring",
      "venting",
      "hazardous air pollutants",
      "air toxics",
      "indoor air quality",
      "ambient air quality",
      "emission control",
      "scrubber",
      "electrostatic precipitator",
      "pollution control",
      "air monitoring"
    ],
    "Water & Wastewater": [
      "water",
      "water consumption",
      "water use",
      "water usage",
      "water withdrawal",
      "water abstraction",
      "freshwater",
      "fresh water",
      "surface water",
      "groundwater",
      "ground water",
      "aquifer",
      "water scarcity",
      "water stress",
      "water stressed",
      "water security",
      "water risk",
      "water footprint",
      "water intensity",
      "water efficiency",
      "water conservation",
      "water management",
      "water stewardship",
      "responsible water use",
      "water recycling",
      "recycled water",
      "water reuse",
      "reused water",
      "rainwater harvesting",
      "rain water harvesting",
      "wastewater",
      "waste water",
      "wastewater treatment",
      "effluent",
      "effluents",
      "effluent treatment",
      "effluent treatment plant",
      "etp",
      "sewage",
      "sewage treatment",
      "sewage treatment plant",
      "stp",
      "zero liquid discharge",
      "zld",
      "desalination",
      "water discharge",
      "industrial wastewater",
      "water pollution",
      "groundwater pollution",
      "water quality",
      "biochemical oxygen demand",
      "bod",
      "chemical oxygen demand",
      "cod",
      "total dissolved solids",
      "tds",
      "water replenishment",
      "watershed",
      "river basin",
      "catchment area"
    ],
    "Waste Management": [
      "waste",
      "waste management",
      "waste generation",
      "waste reduction",
      "waste minimization",
      "waste minimisation",
      "waste disposal",
      "solid waste",
      "municipal waste",
      "industrial waste",
      "hazardous waste",
      "non hazardous waste",
      "non-hazardous waste",
      "toxic waste",
      "electronic waste",
      "e-waste",
      "ewaste",
      "plastic waste",
      "food waste",
      "construction waste",
      "demolition waste",
      "biomedical waste",
      "medical waste",
      "chemical waste",
      "radioactive waste",
      "landfill",
      "landfilling",
      "landfill diversion",
      "incineration",
      "composting",
      "recycling",
      "recycled",
      "recyclable",
      "material recovery",
      "waste recovery",
      "waste treatment",
      "waste segregation",
      "source segregation",
      "zero waste",
      "zero waste to landfill",
      "waste-to-energy",
      "waste to energy",
      "resource recovery",
      "industrial symbiosis"
    ],
    "Circular Economy": [
      "circular economy",
      "circularity",
      "circular business model",
      "closed loop",
      "closed-loop",
      "closed loop recycling",
      "resource efficiency",
      "material efficiency",
      "resource productivity",
      "reduce reuse recycle",
      "reduce",
      "reuse",
      "recycle",
      "repair",
      "refurbish",
      "refurbishment",
      "remanufacture",
      "remanufacturing",
      "recovery",
      "recycled content",
      "post consumer recycled",
      "post-consumer recycled",
      "pcr",
      "secondary raw materials",
      "material recovery",
      "upcycling",
      "downcycling",
      "product longevity",
      "product durability",
      "product life extension",
      "reuse model",
      "sharing economy",
      "product as a service",
      "take back program",
      "take-back programme",
      "extended producer responsibility",
      "epr",
      "design for recycling",
      "design for disassembly",
      "circular design",
      "circular packaging",
      "material circularity"
    ],
    "Biodiversity & Ecosystems": [
      "biodiversity",
      "biological diversity",
      "ecosystem",
      "ecosystems",
      "ecosystem services",
      "ecosystem restoration",
      "ecosystem protection",
      "ecosystem conservation",
      "ecological impact",
      "ecological impacts",
      "ecology",
      "habitat",
      "habitat protection",
      "habitat conservation",
      "habitat restoration",
      "habitat destruction",
      "habitat loss",
      "species",
      "species diversity",
      "threatened species",
      "endangered species",
      "protected species",
      "native species",
      "invasive species",
      "flora",
      "fauna",
      "wildlife",
      "wildlife conservation",
      "nature conservation",
      "nature protection",
      "nature positive",
      "nature-positive",
      "biodiversity loss",
      "biodiversity impact",
      "biodiversity conservation",
      "biodiversity restoration",
      "biodiversity offset",
      "protected area",
      "protected areas",
      "conservation area",
      "ecologically sensitive area",
      "wetland",
      "wetlands",
      "forest ecosystem",
      "pollinator",
      "reforestation",
      "afforestation",
      "restoration ecology"
    ],
    "Land Use & Deforestation": [
      "land use",
      "land-use change",
      "land use change",
      "land degradation",
      "soil degradation",
      "soil erosion",
      "soil conservation",
      "soil quality",
      "soil health",
      "land restoration",
      "land rehabilitation",
      "deforestation",
      "forest loss",
      "forest degradation",
      "forest conservation",
      "forest protection",
      "sustainable forestry",
      "responsible forestry",
      "forest management",
      "reforestation",
      "afforestation",
      "tree planting",
      "tree plantation",
      "green cover",
      "forest cover",
      "no deforestation",
      "zero deforestation",
      "deforestation free",
      "deforestation-free",
      "peatland",
      "peatlands",
      "land conversion",
      "land clearing",
      "desertification",
      "brownfield",
      "greenfield",
      "site restoration",
      "mine rehabilitation"
    ],
    "Natural Resources & Materials": [
      "natural resources",
      "resource conservation",
      "resource management",
      "resource use",
      "raw materials",
      "raw material consumption",
      "material consumption",
      "material use",
      "material sourcing",
      "responsible sourcing",
      "sustainable sourcing",
      "responsible materials",
      "sustainable materials",
      "recycled materials",
      "renewable materials",
      "bio based materials",
      "bio-based materials",
      "critical minerals",
      "critical raw materials",
      "mineral resources",
      "mining",
      "mineral extraction",
      "resource extraction",
      "natural resource extraction",
      "timber",
      "sustainable timber",
      "certified wood",
      "fsc",
      "forest stewardship council",
      "pulp",
      "paper consumption",
      "resource depletion",
      "resource scarcity",
      "material footprint"
    ],
    "Chemicals & Hazardous Materials": [
      "hazardous material",
      "hazardous materials",
      "hazardous substance",
      "hazardous substances",
      "hazardous chemical",
      "hazardous chemicals",
      "toxic chemical",
      "toxic chemicals",
      "chemical safety",
      "chemical management",
      "chemical pollution",
      "toxic substance",
      "toxics",
      "persistent organic pollutants",
      "pop",
      "pops",
      "heavy metals",
      "mercury",
      "lead",
      "cadmium",
      "arsenic",
      "chromium",
      "pfas",
      "perfluoroalkyl",
      "polyfluoroalkyl",
      "pesticide",
      "pesticides",
      "herbicide",
      "herbicides",
      "fertilizer",
      "fertiliser",
      "agrochemical",
      "agrochemicals",
      "solvent",
      "asbestos",
      "formaldehyde",
      "chlorofluorocarbon",
      "cfc",
      "cfcs",
      "ozone depleting substances",
      "ods",
      "chemical spill",
      "oil spill",
      "spill prevention",
      "material safety data sheet",
      "msds",
      "safety data sheet",
      "sds"
    ],
    "Environmental Compliance": [
      "environmental compliance",
      "environmental regulation",
      "environmental regulations",
      "environmental law",
      "environmental legislation",
      "environmental permit",
      "environmental permits",
      "environmental clearance",
      "environmental approval",
      "environmental authorization",
      "environmental licence",
      "environmental license",
      "pollution control board",
      "environmental standard",
      "environmental standards",
      "emission standard",
      "emission limit",
      "discharge limit",
      "regulatory compliance",
      "environmental violation",
      "environmental breach",
      "environmental fine",
      "environmental penalty",
      "environmental litigation",
      "environmental liability",
      "environmental audit",
      "compliance audit",
      "environmental monitoring",
      "consent to operate",
      "consent to establish",
      "environmental impact assessment",
      "eia",
      "environmental due diligence"
    ],
    "Environmental Management": [
      "environmental management",
      "environment management system",
      "environmental management system",
      "ems",
      "iso 14001",
      "environmental policy",
      "environment policy",
      "environmental strategy",
      "environmental target",
      "environmental objective",
      "environmental performance",
      "environmental responsibility",
      "environmental stewardship",
      "environmental sustainability",
      "environmental protection",
      "environmental conservation",
      "environmental footprint",
      "environmental risk",
      "environmental assessment",
      "environmental audit",
      "environmental monitoring",
      "environmental indicator",
      "environmental kpi",
      "environmental management plan",
      "emp",
      "pollution prevention",
      "precautionary principle",
      "environmental certification",
      "eco certification",
      "green certification"
    ],
    "Climate Risk & Resilience": [
      "climate risk",
      "climate risks",
      "physical climate risk",
      "transition risk",
      "climate transition risk",
      "acute climate risk",
      "chronic climate risk",
      "climate vulnerability",
      "climate exposure",
      "climate resilience",
      "climate resilient",
      "climate adaptation",
      "adaptation plan",
      "resilience plan",
      "climate scenario",
      "scenario analysis",
      "climate scenario analysis",
      "climate stress test",
      "extreme weather",
      "extreme heat",
      "heatwave",
      "heat wave",
      "flood",
      "flooding",
      "drought",
      "wildfire",
      "forest fire",
      "cyclone",
      "hurricane",
      "storm",
      "sea level rise",
      "coastal flooding",
      "water stress",
      "climate hazard",
      "climate disaster",
      "natural disaster",
      "disaster resilience",
      "business continuity climate",
      "adaptation finance"
    ],
    "Green Buildings & Infrastructure": [
      "green building",
      "green buildings",
      "sustainable building",
      "sustainable buildings",
      "green infrastructure",
      "sustainable infrastructure",
      "low carbon building",
      "net zero building",
      "net zero energy building",
      "energy efficient building",
      "building efficiency",
      "green construction",
      "sustainable construction",
      "green architecture",
      "leed",
      "breeam",
      "igbc",
      "griha",
      "green roof",
      "cool roof",
      "passive design",
      "smart building",
      "building energy management",
      "embodied carbon",
      "low carbon cement",
      "green cement",
      "sustainable concrete",
      "recycled construction material",
      "climate resilient infrastructure"
    ],
    "Sustainable Transportation": [
      "sustainable transport",
      "sustainable transportation",
      "green mobility",
      "clean mobility",
      "electric vehicle",
      "electric vehicles",
      "ev",
      "evs",
      "electric mobility",
      "e-mobility",
      "zero emission vehicle",
      "zero-emission vehicle",
      "zev",
      "hybrid vehicle",
      "plug in hybrid",
      "plug-in hybrid",
      "phev",
      "battery electric vehicle",
      "bev",
      "fuel cell vehicle",
      "hydrogen vehicle",
      "ev charging",
      "charging infrastructure",
      "charging station",
      "vehicle electrification",
      "fleet electrification",
      "green fleet",
      "sustainable aviation fuel",
      "saf",
      "biofuel",
      "biodiesel",
      "ethanol fuel",
      "public transport",
      "mass transit",
      "rail transport",
      "low carbon transport",
      "transport emissions",
      "vehicle emissions",
      "fuel economy"
    ],
    "Clean Technology": [
      "clean technology",
      "clean technologies",
      "cleantech",
      "green technology",
      "green technologies",
      "climate technology",
      "climate tech",
      "environmental technology",
      "low carbon technology",
      "carbon capture technology",
      "clean energy technology",
      "energy storage technology",
      "battery technology",
      "smart grid technology",
      "hydrogen technology",
      "electrolyzer",
      "electrolyser",
      "fuel cell",
      "heat pump",
      "energy efficient technology",
      "pollution control technology",
      "water treatment technology",
      "waste treatment technology",
      "recycling technology",
      "carbon removal technology",
      "direct air capture",
      "green innovation",
      "eco innovation",
      "environmental innovation",
      "sustainable technology"
    ],
    "Sustainable Agriculture & Food": [
      "sustainable agriculture",
      "sustainable farming",
      "regenerative agriculture",
      "regenerative farming",
      "organic farming",
      "organic agriculture",
      "precision agriculture",
      "climate smart agriculture",
      "climate-smart agriculture",
      "agroforestry",
      "soil health",
      "soil carbon",
      "crop rotation",
      "cover crops",
      "water efficient agriculture",
      "drip irrigation",
      "micro irrigation",
      "sustainable irrigation",
      "pesticide reduction",
      "fertilizer efficiency",
      "fertiliser efficiency",
      "sustainable food",
      "sustainable food system",
      "food waste",
      "responsible agriculture",
      "sustainable livestock",
      "livestock emissions",
      "methane reduction",
      "sustainable fisheries",
      "sustainable aquaculture",
      "deforestation free agriculture",
      "certified sustainable agriculture"
    ],
    "Oceans & Marine Environment": [
      "ocean",
      "oceans",
      "marine",
      "marine environment",
      "marine ecosystem",
      "marine biodiversity",
      "ocean biodiversity",
      "ocean conservation",
      "marine conservation",
      "ocean pollution",
      "marine pollution",
      "plastic pollution",
      "ocean plastic",
      "marine litter",
      "microplastic",
      "microplastics",
      "coral reef",
      "coral reefs",
      "mangrove",
      "mangroves",
      "coastal ecosystem",
      "coastal conservation",
      "blue economy",
      "blue carbon",
      "ocean acidification",
      "marine protected area",
      "sustainable fishing",
      "overfishing",
      "sustainable fisheries",
      "bycatch",
      "oil spill",
      "marine restoration"
    ],
    "Carbon Markets & Climate Finance": [
      "carbon market",
      "carbon markets",
      "carbon trading",
      "emissions trading",
      "emission trading scheme",
      "ets",
      "carbon credit",
      "carbon credits",
      "carbon offset",
      "carbon offsets",
      "carbon allowance",
      "carbon certificate",
      "voluntary carbon market",
      "compliance carbon market",
      "carbon price",
      "carbon pricing",
      "carbon tax",
      "green finance",
      "climate finance",
      "sustainable finance",
      "green bond",
      "green bonds",
      "climate bond",
      "sustainability bond",
      "sustainability linked bond",
      "sustainability-linked bond",
      "green loan",
      "sustainability linked loan",
      "transition finance",
      "climate fund",
      "green investment",
      "climate investment",
      "environmental finance"
    ],
    "Product Lifecycle & Eco-design": [
      "life cycle assessment",
      "lifecycle assessment",
      "lca",
      "life cycle analysis",
      "cradle to grave",
      "cradle-to-grave",
      "cradle to cradle",
      "cradle-to-cradle",
      "product lifecycle",
      "product life cycle",
      "eco design",
      "eco-design",
      "ecodesign",
      "sustainable design",
      "green design",
      "design for environment",
      "environmentally friendly product",
      "eco friendly product",
      "eco-friendly product",
      "green product",
      "sustainable product",
      "low carbon product",
      "environmental product declaration",
      "epd",
      "product carbon footprint",
      "environmental footprint",
      "sustainable packaging",
      "eco friendly packaging",
      "recyclable packaging",
      "reusable packaging",
      "biodegradable packaging",
      "compostable packaging",
      "plastic free",
      "single use plastic",
      "single-use plastic",
      "packaging reduction"
    ],
    "Sustainable Supply Chain": [
      "sustainable supply chain",
      "green supply chain",
      "responsible supply chain",
      "supply chain sustainability",
      "supplier sustainability",
      "supplier environmental assessment",
      "environmental supplier assessment",
      "supplier emissions",
      "supply chain emissions",
      "scope 3 emissions",
      "upstream emissions",
      "downstream emissions",
      "responsible sourcing",
      "sustainable sourcing",
      "green procurement",
      "sustainable procurement",
      "environmental procurement",
      "supplier environmental performance",
      "supplier climate target",
      "supplier decarbonization",
      "supplier decarbonisation",
      "low carbon supply chain",
      "supply chain carbon footprint",
      "traceability",
      "raw material traceability",
      "deforestation free supply chain",
      "sustainable logistics",
      "green logistics",
      "logistics emissions"
    ],
    "Environmental Reporting & Metrics": [
      "environmental reporting",
      "sustainability reporting",
      "esg reporting",
      "environmental disclosure",
      "climate disclosure",
      "climate reporting",
      "carbon disclosure",
      "sustainability disclosure",
      "environmental kpi",
      "environmental metric",
      "environmental metrics",
      "sustainability metric",
      "climate metric",
      "ghg protocol",
      "greenhouse gas protocol",
      "gri",
      "global reporting initiative",
      "sasb",
      "sustainability accounting standards board",
      "issb",
      "international sustainability standards board",
      "ifrs s1",
      "ifrs s2",
      "cdp",
      "tcfd",
      "task force on climate-related financial disclosures",
      "tnfd",
      "taskforce on nature-related financial disclosures",
      "csrd",
      "esrs",
      "science based targets",
      "sbti",
      "double materiality",
      "climate materiality",
      "environmental materiality",
      "environmental performance indicator"
    ],
    "Environmental Remediation": [
      "environmental remediation",
      "site remediation",
      "soil remediation",
      "groundwater remediation",
      "contaminated land",
      "land contamination",
      "soil contamination",
      "groundwater contamination",
      "site contamination",
      "cleanup",
      "environmental cleanup",
      "pollution cleanup",
      "spill response",
      "oil spill response",
      "spill remediation",
      "bioremediation",
      "phytoremediation",
      "mine closure",
      "mine reclamation",
      "mine rehabilitation",
      "land reclamation",
      "restoration",
      "ecological restoration",
      "site restoration",
      "brownfield remediation",
      "environmental restoration"
    ]
  },
  "social_keywords": {
    "CSR and community investment": [
      "CSR",
      "corporate social responsibility",
      "social responsibility",
      "social impact",
      "social initiative",
      "social welfare",
      "community development",
      "corporate citizenship",
      "shared value",
      "philanthropy",
      "social investment",
      "donation",
      "donations",
      "grant",
      "grants",
      "fund allocation",
      "contribution",
      "charitable contribution",
      "sponsorship",
      "corpus fund",
      "trust fund",
      "CSR spend",
      "CSR expenditure",
      "CSR budget",
      "CSR committee",
      "CSR policy",
      "CSR report",
      "annual CSR report",
      "Section 135",
      "Companies Act 2013",
      "Schedule VII",
      "CSR activities",
      "CSR projects",
      "partner NGO",
      "implementing agency",
      "CSR partner"
    ],
    "Education and skills": [
      "education",
      "literacy",
      "adult literacy",
      "school infrastructure",
      "digital literacy",
      "e-learning",
      "vocational training",
      "skill development",
      "skilling",
      "scholarship program",
      "scholarship",
      "scholarships",
      "mid-day meal",
      "teacher training",
      "STEM education",
      "girl child education",
      "dropout prevention",
      "learning outcomes",
      "education for all",
      "mentorship program",
      "university partnership",
      "research grant",
      "fellowship"
    ],
    "Health and nutrition": [
      "healthcare",
      "health camp",
      "medical camp",
      "free medical checkup",
      "immunization",
      "vaccination drive",
      "maternal health",
      "child health",
      "nutrition program",
      "malnutrition",
      "sanitation",
      "hygiene awareness",
      "mental health",
      "wellness program",
      "blood donation camp",
      "hospital infrastructure",
      "telemedicine",
      "healthcare access",
      "WASH",
      "water sanitation hygiene"
    ],
    "Rural development and livelihoods": [
      "rural development",
      "village adoption",
      "community center",
      "infrastructure development",
      "drinking water",
      "clean water access",
      "sanitation facilities",
      "livelihood",
      "livelihood generation",
      "self-help group",
      "SHG",
      "microfinance",
      "financial inclusion",
      "farmer welfare",
      "agricultural training",
      "cooperative",
      "slum development",
      "urban poor",
      "community outreach",
      "grassroots initiative",
      "capacity building",
      "local employment"
    ],
    "Inclusion and worker welfare": [
      "women empowerment",
      "gender equality",
      "women entrepreneurship",
      "girl child",
      "widow support",
      "differently abled",
      "persons with disabilities",
      "PwD",
      "marginalized communities",
      "underprivileged",
      "tribal welfare",
      "SC/ST welfare",
      "minority welfare",
      "LGBTQ inclusion",
      "senior citizen welfare",
      "orphan care",
      "vulnerable groups",
      "inclusive growth",
      "equal opportunity",
      "disability support",
      "employee welfare",
      "workplace diversity",
      "diversity and inclusion",
      "D&I",
      "fair wages",
      "labor welfare",
      "occupational health and safety",
      "employee engagement in CSR"
    ],
    "Environment and climate initiatives": [
      "tree plantation",
      "afforestation",
      "reforestation",
      "environmental conservation",
      "waste management",
      "water conservation",
      "renewable energy access",
      "clean energy",
      "carbon footprint reduction",
      "biodiversity conservation",
      "wildlife protection",
      "pollution control",
      "climate action",
      "sustainable agriculture",
      "eco-friendly initiative",
      "green initiative",
      "solar energy project",
      "rainwater harvesting"
    ],
    "Volunteering and relief": [
      "employee volunteering",
      "volunteer program",
      "volunteering hours",
      "giving campaign",
      "payroll giving",
      "disaster relief",
      "disaster management",
      "flood relief",
      "earthquake relief",
      "humanitarian aid",
      "emergency response",
      "relief camp",
      "rehabilitation",
      "resettlement",
      "calamity relief",
      "COVID relief",
      "pandemic response",
      "food distribution",
      "essential supplies",
      "relief fund",
      "disaster relief fund"
    ],
    "Impact measurement and reporting": [
      "social audit",
      "impact assessment",
      "beneficiaries",
      "beneficiaries reached",
      "lives touched",
      "lives impacted",
      "outreach program",
      "Sustainable Development Goals",
      "SDG",
      "Global Reporting Initiative",
      "GRI",
      "BRSR",
      "Business Responsibility and Sustainability Report",
      "social license",
      "stakeholder engagement",
      "triple bottom line",
      "social stewardship"
    ],
    "Other user supplied social terms": [
      "sustainability",
      "ESG",
      "environmental social governance",
      "social good",
      "public good",
      "corporate philanthropy",
      "giving back",
      "impact investing",
      "funding",
      "endowment",
      "seed funding",
      "matching gift",
      "in-kind donation",
      "pro bono",
      "subsidy",
      "Business Responsibility and Sustainability Report"
    ]
  },
  "governance_sectors": [
    "Corporate Governance",
    "Board Structure & Independence",
    "Board Diversity & Expertise",
    "Board Committees",
    "Executive Management",
    "Executive Compensation",
    "Business Ethics & Integrity",
    "Anti-Bribery & Anti-Corruption",
    "Fraud & Financial Crime",
    "Whistleblower & Grievance Mechanisms",
    "Risk Management",
    "Enterprise Risk Management",
    "Internal Controls",
    "Audit & Assurance",
    "Regulatory Compliance",
    "Legal & Litigation",
    "Shareholder Rights",
    "Ownership & Control",
    "Related Party Transactions",
    "Transparency & Disclosure",
    "ESG Governance & Oversight",
    "Tax Governance",
    "Political Contributions & Lobbying",
    "Competition & Antitrust",
    "Cybersecurity Governance",
    "Data Privacy & Protection",
    "AI & Technology Governance",
    "Supply Chain Governance",
    "Human Rights Governance",
    "Crisis & Incident Management",
    "Systemic Risk",
    "Business Continuity",
    "Conflicts of Interest",
    "Insider Trading",
    "Information Security",
    "Intellectual Property Governance",
    "Responsible Investment Governance",
    "Corporate Policies & Codes",
    "Governance Ratings & Frameworks",
    "India Corporate Governance"
  ],
  "governance_keywords": {
    "Corporate Governance": [
      "corporate governance",
      "governance",
      "good governance",
      "governance framework",
      "governance structure",
      "governance model",
      "governance system",
      "governance practices",
      "governance policy",
      "governance policies",
      "governance principles",
      "governance standards",
      "governance mechanism",
      "governance mechanisms",
      "governance oversight",
      "corporate oversight",
      "management oversight",
      "governance responsibility",
      "governance accountability",
      "corporate accountability",
      "management accountability",
      "accountability framework",
      "fiduciary responsibility",
      "fiduciary duty",
      "fiduciary duties",
      "stewardship",
      "corporate stewardship",
      "responsible governance",
      "ethical governance",
      "governance effectiveness",
      "governance performance",
      "governance review",
      "governance assessment",
      "governance evaluation",
      "corporate governance report",
      "governance disclosure",
      "governance disclosures",
      "corporate governance code",
      "code of corporate governance"
    ],
    "Board Structure & Independence": [
      "board of directors",
      "board",
      "board structure",
      "board composition",
      "board size",
      "board member",
      "board members",
      "director",
      "directors",
      "independent director",
      "independent directors",
      "independent board member",
      "board independence",
      "director independence",
      "non-executive director",
      "non executive director",
      "non-executive directors",
      "executive director",
      "executive directors",
      "outside director",
      "lead independent director",
      "presiding director",
      "chairman",
      "chairperson",
      "board chair",
      "independent chair",
      "executive chairman",
      "non-executive chairman",
      "chair and ceo separation",
      "ceo chair separation",
      "separation of chairman and ceo",
      "board leadership structure",
      "board tenure",
      "director tenure",
      "average board tenure",
      "board refreshment",
      "director rotation",
      "retirement by rotation",
      "director nomination",
      "director appointment",
      "board appointment",
      "director re-election",
      "director reelection",
      "board election",
      "board effectiveness",
      "board evaluation",
      "board performance evaluation",
      "annual board evaluation",
      "director evaluation",
      "board meeting",
      "board meetings",
      "board attendance",
      "director attendance",
      "board meeting attendance",
      "quorum",
      "board calendar",
      "board responsibility",
      "board responsibilities"
    ],
    "Board Diversity & Expertise": [
      "board diversity",
      "board gender diversity",
      "gender diversity on board",
      "women directors",
      "woman director",
      "female director",
      "female directors",
      "diverse board",
      "board skills",
      "board expertise",
      "director expertise",
      "board competency",
      "board competencies",
      "skills matrix",
      "board skills matrix",
      "director skills matrix",
      "board experience",
      "industry experience",
      "financial expertise",
      "financial expert",
      "audit committee financial expert",
      "technology expertise",
      "cybersecurity expertise",
      "sustainability expertise",
      "esg expertise",
      "climate expertise",
      "risk expertise",
      "legal expertise",
      "international experience",
      "board qualification",
      "director qualification",
      "director qualifications",
      "board succession",
      "director succession",
      "board succession planning"
    ],
    "Board Committees": [
      "board committee",
      "board committees",
      "audit committee",
      "risk committee",
      "risk management committee",
      "nomination committee",
      "nominating committee",
      "nomination and remuneration committee",
      "remuneration committee",
      "compensation committee",
      "esg committee",
      "sustainability committee",
      "corporate responsibility committee",
      "csr committee",
      "stakeholder relationship committee",
      "stakeholders relationship committee",
      "ethics committee",
      "compliance committee",
      "finance committee",
      "investment committee",
      "technology committee",
      "cybersecurity committee",
      "strategy committee",
      "executive committee",
      "board subcommittee",
      "committee charter",
      "audit committee charter",
      "committee independence",
      "committee member",
      "committee chairman",
      "committee chair",
      "committee meeting",
      "committee attendance"
    ],
    "Executive Management": [
      "executive management",
      "senior management",
      "senior leadership",
      "leadership team",
      "management team",
      "key management personnel",
      "kmp",
      "chief executive officer",
      "ceo",
      "chief financial officer",
      "cfo",
      "chief operating officer",
      "coo",
      "company secretary",
      "chief compliance officer",
      "cco",
      "chief risk officer",
      "cro",
      "chief audit executive",
      "chief sustainability officer",
      "cso",
      "chief information security officer",
      "ciso",
      "executive leadership",
      "leadership accountability",
      "management responsibility",
      "management oversight",
      "management succession",
      "succession planning",
      "ceo succession",
      "leadership succession",
      "key person risk",
      "management continuity"
    ],
    "Executive Compensation": [
      "executive compensation",
      "executive remuneration",
      "director remuneration",
      "management compensation",
      "ceo compensation",
      "ceo pay",
      "executive pay",
      "management remuneration",
      "remuneration policy",
      "compensation policy",
      "pay policy",
      "salary",
      "executive salary",
      "performance bonus",
      "annual bonus",
      "incentive compensation",
      "incentive pay",
      "variable compensation",
      "variable pay",
      "short term incentive",
      "short-term incentive",
      "sti",
      "long term incentive",
      "long-term incentive",
      "lti",
      "stock compensation",
      "stock based compensation",
      "stock-based compensation",
      "equity compensation",
      "share based payment",
      "share-based payment",
      "stock options",
      "employee stock option",
      "esop",
      "restricted stock",
      "restricted stock unit",
      "rsu",
      "performance shares",
      "performance stock units",
      "pay for performance",
      "performance linked compensation",
      "performance-linked remuneration",
      "esg linked compensation",
      "esg-linked compensation",
      "sustainability linked compensation",
      "climate linked compensation",
      "clawback",
      "clawback policy",
      "malus",
      "golden parachute",
      "severance payment",
      "pay ratio",
      "ceo pay ratio",
      "median employee pay",
      "remuneration ratio",
      "say on pay"
    ],
    "Business Ethics & Integrity": [
      "business ethics",
      "corporate ethics",
      "ethical business",
      "ethical conduct",
      "ethical behaviour",
      "ethical behavior",
      "integrity",
      "business integrity",
      "corporate integrity",
      "ethical standards",
      "ethical principles",
      "ethical culture",
      "ethics policy",
      "ethics programme",
      "ethics program",
      "code of ethics",
      "code of conduct",
      "business conduct",
      "standards of conduct",
      "professional conduct",
      "responsible business conduct",
      "ethical decision making",
      "ethical leadership",
      "integrity training",
      "ethics training",
      "code of conduct training",
      "misconduct",
      "corporate misconduct",
      "ethical violation",
      "ethical breach",
      "misrepresentation",
      "negligence",
      "dishonesty",
      "corporate wrongdoing"
    ],
    "Anti-Bribery & Anti-Corruption": [
      "corruption",
      "anti-corruption",
      "anti corruption",
      "anticorruption",
      "bribery",
      "anti-bribery",
      "anti bribery",
      "bribe",
      "bribes",
      "bribery risk",
      "corruption risk",
      "corrupt practices",
      "corrupt practice",
      "improper payment",
      "improper payments",
      "facilitation payment",
      "facilitation payments",
      "kickback",
      "kickbacks",
      "payoff",
      "illegal payment",
      "illicit payment",
      "commercial bribery",
      "public sector bribery",
      "foreign bribery",
      "anti-corruption policy",
      "anti-bribery policy",
      "anti-corruption programme",
      "anti-bribery programme",
      "anti-corruption training",
      "anti-bribery training",
      "corruption prevention",
      "bribery prevention",
      "corruption investigation",
      "bribery investigation",
      "corruption allegation",
      "bribery allegation",
      "zero tolerance corruption",
      "zero tolerance bribery",
      "foreign corrupt practices act",
      "fcpa",
      "uk bribery act"
    ],
    "Fraud & Financial Crime": [
      "fraud",
      "fraudulent activity",
      "fraudulent activities",
      "fraud prevention",
      "fraud detection",
      "fraud investigation",
      "fraud risk",
      "financial fraud",
      "corporate fraud",
      "accounting fraud",
      "securities fraud",
      "procurement fraud",
      "expense fraud",
      "asset misappropriation",
      "misappropriation",
      "embezzlement",
      "money laundering",
      "anti-money laundering",
      "anti money laundering",
      "aml",
      "financial crime",
      "financial crimes",
      "financial crime prevention",
      "terrorist financing",
      "terrorism financing",
      "counter terrorist financing",
      "ctf",
      "know your customer",
      "know-your-customer",
      "kyc",
      "customer due diligence",
      "cdd",
      "enhanced due diligence",
      "edd",
      "suspicious transaction",
      "suspicious activity",
      "suspicious activity report",
      "sar",
      "sanctions compliance",
      "economic sanctions",
      "trade sanctions"
    ],
    "Whistleblower & Grievance Mechanisms": [
      "whistleblower",
      "whistle blower",
      "whistleblowing",
      "whistle blowing",
      "whistleblower policy",
      "whistleblower programme",
      "whistleblower program",
      "whistleblower protection",
      "protected disclosure",
      "protected disclosures",
      "vigil mechanism",
      "vigilance mechanism",
      "ethics hotline",
      "compliance hotline",
      "integrity hotline",
      "speak up",
      "speak-up",
      "speak up culture",
      "report misconduct",
      "misconduct reporting",
      "anonymous reporting",
      "confidential reporting",
      "grievance mechanism",
      "grievance mechanisms",
      "grievance procedure",
      "complaint mechanism",
      "complaints mechanism",
      "complaint channel",
      "employee complaint",
      "ethics complaint",
      "retaliation",
      "non-retaliation",
      "non retaliation",
      "anti-retaliation",
      "whistleblower retaliation",
      "grievance redressal",
      "grievance redressal mechanism"
    ],
    "Risk Management": [
      "risk management",
      "risk governance",
      "risk oversight",
      "risk framework",
      "risk management framework",
      "risk policy",
      "risk management policy",
      "risk appetite",
      "risk appetite framework",
      "risk tolerance",
      "risk capacity",
      "risk identification",
      "risk assessment",
      "risk analysis",
      "risk evaluation",
      "risk monitoring",
      "risk mitigation",
      "risk treatment",
      "risk control",
      "risk controls",
      "risk register",
      "risk profile",
      "risk exposure",
      "risk indicator",
      "key risk indicator",
      "kri",
      "risk culture",
      "risk owner",
      "risk ownership",
      "emerging risk",
      "emerging risks",
      "strategic risk",
      "operational risk",
      "financial risk",
      "reputational risk",
      "compliance risk",
      "legal risk",
      "regulatory risk",
      "technology risk",
      "cyber risk",
      "third party risk",
      "third-party risk",
      "concentration risk"
    ],
    "Enterprise Risk Management": [
      "enterprise risk management",
      "erm",
      "integrated risk management",
      "enterprise risk",
      "enterprise-wide risk",
      "enterprise wide risk",
      "risk management system",
      "risk management process",
      "risk governance framework",
      "risk committee",
      "chief risk officer",
      "risk dashboard",
      "risk heat map",
      "risk heatmap",
      "risk matrix",
      "risk scenario",
      "scenario analysis",
      "stress testing",
      "stress test",
      "sensitivity analysis",
      "risk aggregation",
      "risk correlation",
      "risk interdependency",
      "risk taxonomy",
      "risk reporting",
      "board risk oversight",
      "three lines model",
      "three lines of defense",
      "three lines of defence",
      "first line of defense",
      "second line of defense",
      "third line of defense",
      "coso erm",
      "iso 31000"
    ],
    "Internal Controls": [
      "internal control",
      "internal controls",
      "internal control system",
      "internal control framework",
      "internal financial controls",
      "ifc",
      "internal controls over financial reporting",
      "icfr",
      "control environment",
      "control activity",
      "control activities",
      "control testing",
      "control effectiveness",
      "control deficiency",
      "control deficiencies",
      "significant deficiency",
      "material weakness",
      "key control",
      "key controls",
      "preventive control",
      "preventative control",
      "detective control",
      "manual control",
      "automated control",
      "financial control",
      "operational control",
      "access control",
      "segregation of duties",
      "separation of duties",
      "sod",
      "maker checker",
      "maker-checker",
      "authorization control",
      "approval control",
      "management control",
      "coso framework",
      "control self assessment",
      "control self-assessment"
    ],
    "Audit & Assurance": [
      "audit",
      "auditing",
      "internal audit",
      "external audit",
      "statutory audit",
      "independent audit",
      "financial audit",
      "compliance audit",
      "operational audit",
      "forensic audit",
      "information systems audit",
      "it audit",
      "esg audit",
      "sustainability assurance",
      "external assurance",
      "independent assurance",
      "limited assurance",
      "reasonable assurance",
      "audit committee",
      "audit plan",
      "audit programme",
      "audit program",
      "audit scope",
      "audit finding",
      "audit findings",
      "audit observation",
      "audit observations",
      "audit recommendation",
      "audit report",
      "auditor",
      "auditors",
      "external auditor",
      "statutory auditor",
      "independent auditor",
      "auditor independence",
      "audit independence",
      "audit rotation",
      "auditor rotation",
      "audit partner rotation",
      "audit quality",
      "audit opinion",
      "qualified opinion",
      "unqualified opinion",
      "adverse opinion",
      "disclaimer of opinion",
      "key audit matter",
      "key audit matters",
      "kam",
      "going concern",
      "assurance provider"
    ],
    "Regulatory Compliance": [
      "compliance",
      "regulatory compliance",
      "legal compliance",
      "statutory compliance",
      "compliance framework",
      "compliance programme",
      "compliance program",
      "compliance policy",
      "compliance management",
      "compliance management system",
      "compliance monitoring",
      "compliance testing",
      "compliance audit",
      "compliance review",
      "regulatory requirement",
      "regulatory requirements",
      "legal requirement",
      "statutory requirement",
      "law and regulation",
      "laws and regulations",
      "regulatory obligation",
      "regulatory obligations",
      "compliance obligation",
      "regulatory authority",
      "regulatory body",
      "regulator",
      "regulators",
      "regulatory filing",
      "statutory filing",
      "regulatory approval",
      "regulatory investigation",
      "regulatory action",
      "regulatory enforcement",
      "compliance breach",
      "compliance violation",
      "regulatory breach",
      "regulatory violation",
      "non-compliance",
      "noncompliance",
      "compliance training",
      "compliance officer",
      "chief compliance officer"
    ],
    "Legal & Litigation": [
      "litigation",
      "legal proceeding",
      "legal proceedings",
      "legal action",
      "legal actions",
      "lawsuit",
      "lawsuits",
      "court case",
      "legal case",
      "pending litigation",
      "material litigation",
      "legal dispute",
      "legal disputes",
      "regulatory proceeding",
      "administrative proceeding",
      "arbitration",
      "mediation",
      "legal claim",
      "legal claims",
      "legal liability",
      "contingent liability",
      "legal exposure",
      "legal provision",
      "settlement",
      "legal settlement",
      "consent decree",
      "court order",
      "injunction",
      "penalty",
      "penalties",
      "fine",
      "fines",
      "monetary penalty",
      "regulatory penalty",
      "sanction",
      "sanctions",
      "legal notice",
      "show cause notice",
      "investigation",
      "regulatory investigation"
    ],
    "Shareholder Rights": [
      "shareholder rights",
      "shareholder protection",
      "minority shareholder",
      "minority shareholders",
      "minority shareholder rights",
      "investor rights",
      "voting rights",
      "shareholder voting",
      "shareholder vote",
      "one share one vote",
      "proxy voting",
      "proxy vote",
      "proxy statement",
      "proxy ballot",
      "annual general meeting",
      "agm",
      "extraordinary general meeting",
      "egm",
      "general meeting",
      "shareholder meeting",
      "shareholder resolution",
      "shareholder proposal",
      "shareholder requisition",
      "special resolution",
      "ordinary resolution",
      "postal ballot",
      "electronic voting",
      "e-voting",
      "evoting",
      "cumulative voting",
      "director election",
      "director re-election",
      "shareholder engagement",
      "investor engagement",
      "investor relations",
      "minority protection",
      "shareholder democracy",
      "shareholder approval"
    ],
    "Ownership & Control": [
      "ownership structure",
      "ownership",
      "shareholding pattern",
      "shareholder structure",
      "shareholding",
      "major shareholder",
      "major shareholders",
      "controlling shareholder",
      "controlling shareholders",
      "controlling interest",
      "control rights",
      "voting control",
      "beneficial ownership",
      "beneficial owner",
      "ultimate beneficial owner",
      "ubo",
      "promoter",
      "promoters",
      "promoter group",
      "promoter shareholding",
      "institutional ownership",
      "foreign institutional ownership",
      "government ownership",
      "state ownership",
      "family ownership",
      "cross ownership",
      "cross shareholding",
      "pyramid ownership",
      "dual class shares",
      "dual-class shares",
      "super voting shares",
      "golden share",
      "ownership concentration",
      "control concentration",
      "free float",
      "public shareholding",
      "pledged shares",
      "share pledge",
      "promoter pledge"
    ],
    "Related Party Transactions": [
      "related party",
      "related parties",
      "related party transaction",
      "related party transactions",
      "rpt",
      "rpts",
      "material related party transaction",
      "related party disclosure",
      "related party approval",
      "related party policy",
      "related party relationship",
      "related entity",
      "related entities",
      "affiliate transaction",
      "affiliated transaction",
      "connected party transaction",
      "connected transaction",
      "intercompany transaction",
      "inter-company transaction",
      "arm length transaction",
      "arm's length transaction",
      "arms length transaction",
      "arm's length basis",
      "transfer pricing",
      "conflict transaction",
      "promoter transaction",
      "director transaction",
      "key management personnel transaction"
    ],
    "Transparency & Disclosure": [
      "transparency",
      "corporate transparency",
      "disclosure",
      "corporate disclosure",
      "public disclosure",
      "financial disclosure",
      "non-financial disclosure",
      "nonfinancial disclosure",
      "mandatory disclosure",
      "voluntary disclosure",
      "material disclosure",
      "timely disclosure",
      "continuous disclosure",
      "regulatory disclosure",
      "stock exchange disclosure",
      "annual report",
      "integrated report",
      "sustainability report",
      "esg report",
      "corporate governance report",
      "financial statement",
      "financial statements",
      "financial reporting",
      "reporting transparency",
      "disclosure controls",
      "disclosure policy",
      "material information",
      "material event",
      "material event disclosure",
      "materiality",
      "materiality assessment",
      "financial transparency",
      "accountability",
      "information transparency",
      "reporting integrity"
    ],
    "ESG Governance & Oversight": [
      "esg governance",
      "esg oversight",
      "board esg oversight",
      "board sustainability oversight",
      "sustainability governance",
      "sustainability oversight",
      "sustainability committee",
      "esg committee",
      "esg responsibility",
      "esg accountability",
      "esg strategy",
      "esg policy",
      "esg framework",
      "esg risk",
      "esg risks",
      "esg risk management",
      "sustainability risk",
      "sustainability risks",
      "board climate oversight",
      "climate governance",
      "climate oversight",
      "climate risk governance",
      "board climate responsibility",
      "environmental oversight",
      "social oversight",
      "responsible business governance",
      "sustainability management",
      "esg management",
      "esg performance",
      "esg targets",
      "esg kpi",
      "sustainability kpi",
      "esg linked remuneration",
      "sustainability linked remuneration",
      "chief sustainability officer"
    ],
    "Tax Governance": [
      "tax governance",
      "tax strategy",
      "tax policy",
      "responsible tax",
      "responsible taxation",
      "tax transparency",
      "tax disclosure",
      "tax compliance",
      "tax risk",
      "tax risk management",
      "tax control framework",
      "tax planning",
      "aggressive tax planning",
      "tax avoidance",
      "tax evasion",
      "tax shelter",
      "tax haven",
      "tax havens",
      "offshore tax",
      "corporate tax",
      "effective tax rate",
      "cash tax",
      "tax paid",
      "income tax paid",
      "country by country reporting",
      "country-by-country reporting",
      "cbcr",
      "base erosion",
      "profit shifting",
      "base erosion and profit shifting",
      "beps",
      "transfer pricing",
      "tax dispute",
      "tax litigation",
      "tax audit",
      "tax authority",
      "tax contribution"
    ],
    "Political Contributions & Lobbying": [
      "political contribution",
      "political contributions",
      "political donation",
      "political donations",
      "political spending",
      "political expenditure",
      "campaign contribution",
      "campaign contributions",
      "corporate political contribution",
      "corporate political spending",
      "lobbying",
      "lobbying activity",
      "lobbying activities",
      "lobbying expenditure",
      "lobbying expenses",
      "lobbying disclosure",
      "lobbyist",
      "public policy engagement",
      "policy advocacy",
      "government relations",
      "government affairs",
      "trade association",
      "trade association membership",
      "industry association",
      "political engagement",
      "regulatory engagement",
      "public affairs",
      "political action committee",
      "pac",
      "political funding",
      "election funding"
    ],
    "Competition & Antitrust": [
      "competition",
      "competitive behaviour",
      "competitive behavior",
      "fair competition",
      "competition law",
      "antitrust",
      "anti-trust",
      "antitrust law",
      "antitrust compliance",
      "anti-competitive",
      "anticompetitive",
      "anti-competitive conduct",
      "anti-competitive practice",
      "competition violation",
      "antitrust violation",
      "price fixing",
      "price-fixing",
      "bid rigging",
      "bid-rigging",
      "collusion",
      "cartel",
      "cartel conduct",
      "market manipulation",
      "market dominance",
      "dominant position",
      "abuse of dominance",
      "monopoly",
      "monopolistic practice",
      "restraint of trade",
      "unfair competition",
      "predatory pricing",
      "market allocation",
      "competition investigation",
      "antitrust investigation",
      "competition authority"
    ],
    "Cybersecurity Governance": [
      "cybersecurity",
      "cyber security",
      "cybersecurity governance",
      "cyber governance",
      "cyber risk",
      "cyber risk management",
      "cybersecurity risk",
      "cybersecurity policy",
      "cybersecurity framework",
      "cybersecurity controls",
      "cybersecurity programme",
      "cybersecurity program",
      "cybersecurity strategy",
      "cybersecurity oversight",
      "board cybersecurity oversight",
      "cybersecurity committee",
      "cybersecurity audit",
      "cybersecurity assessment",
      "cybersecurity incident",
      "cyber incident",
      "cyber attack",
      "cyberattack",
      "ransomware",
      "malware",
      "phishing",
      "security breach",
      "data breach",
      "network security",
      "endpoint security",
      "security operations center",
      "security operations centre",
      "soc",
      "penetration testing",
      "vulnerability assessment",
      "vulnerability management",
      "incident response",
      "cyber incident response",
      "zero trust",
      "cyber resilience",
      "information security management system",
      "isms",
      "iso 27001",
      "nist cybersecurity framework",
      "nist csf",
      "chief information security officer",
      "ciso"
    ],
    "Data Privacy & Protection": [
      "data privacy",
      "privacy",
      "data protection",
      "personal data",
      "personal information",
      "personally identifiable information",
      "pii",
      "sensitive personal data",
      "sensitive data",
      "privacy policy",
      "data protection policy",
      "privacy governance",
      "data governance",
      "data protection officer",
      "dpo",
      "privacy officer",
      "data processing",
      "data processor",
      "data controller",
      "consent management",
      "user consent",
      "data subject",
      "data subject rights",
      "right to access",
      "right to erasure",
      "right to be forgotten",
      "data retention",
      "data minimization",
      "data minimisation",
      "purpose limitation",
      "privacy by design",
      "privacy by default",
      "data breach",
      "privacy breach",
      "data leakage",
      "data leak",
      "gdpr",
      "general data protection regulation",
      "ccpa",
      "digital personal data protection",
      "digital personal data protection act",
      "dpdp",
      "dpdpa",
      "cross border data transfer",
      "data localization",
      "data localisation"
    ],
    "AI & Technology Governance": [
      "ai governance",
      "artificial intelligence governance",
      "responsible ai",
      "ethical ai",
      "trustworthy ai",
      "ai ethics",
      "ai risk",
      "ai risk management",
      "ai oversight",
      "algorithmic governance",
      "algorithm governance",
      "algorithmic accountability",
      "algorithmic bias",
      "ai bias",
      "machine learning governance",
      "model governance",
      "model risk",
      "model risk management",
      "model validation",
      "ai transparency",
      "algorithmic transparency",
      "explainable ai",
      "xai",
      "ai accountability",
      "human oversight",
      "human in the loop",
      "human-in-the-loop",
      "automated decision making",
      "responsible technology",
      "technology governance",
      "technology risk",
      "digital governance",
      "data ethics",
      "ai policy",
      "acceptable use of ai",
      "generative ai governance",
      "genai governance"
    ],
    "Supply Chain Governance": [
      "supply chain governance",
      "supplier governance",
      "supplier management",
      "vendor management",
      "third party management",
      "third-party management",
      "third party risk",
      "third-party risk",
      "third party risk management",
      "vendor risk management",
      "supplier risk",
      "supplier due diligence",
      "vendor due diligence",
      "third party due diligence",
      "supplier audit",
      "vendor audit",
      "supplier assessment",
      "supplier screening",
      "supplier monitoring",
      "supplier compliance",
      "supplier code of conduct",
      "vendor code of conduct",
      "responsible sourcing policy",
      "procurement governance",
      "procurement policy",
      "procurement controls",
      "responsible procurement",
      "sustainable procurement",
      "supply chain transparency",
      "supplier traceability",
      "supply chain traceability",
      "contractor governance"
    ],
    "Human Rights Governance": [
      "human rights governance",
      "human rights policy",
      "human rights due diligence",
      "hrdd"
    ]
  },
  "news_event_terms": {
    "E": [
      "environmental violation",
      "pollution violation",
      "emissions breach",
      "illegal discharge",
      "toxic spill",
      "chemical spill",
      "oil spill",
      "environmental fine",
      "environmental penalty",
      "waste violation",
      "deforestation",
      "habitat destruction",
      "biodiversity damage",
      "water contamination",
      "air pollution",
      "environmental lawsuit",
      "environmental investigation",
      "climate fraud",
      "greenwashing"
    ],
    "S": [
      "worker fatality",
      "workplace death",
      "industrial accident",
      "unsafe working conditions",
      "child labour",
      "forced labour",
      "wage theft",
      "unpaid wages",
      "sexual harassment",
      "discrimination",
      "human rights abuse",
      "community displacement",
      "land rights violation",
      "public health harm",
      "food safety violation",
      "product safety recall",
      "consumer fraud",
      "privacy breach",
      "data breach",
      "social security violation"
    ],
    "G": [
      "bribery",
      "corruption",
      "fraud",
      "accounting fraud",
      "financial misstatement",
      "insider trading",
      "related party abuse",
      "conflict of interest",
      "regulatory fine",
      "regulatory investigation",
      "criminal charges",
      "money laundering",
      "tax evasion",
      "board resignation",
      "auditor resignation",
      "whistleblower retaliation",
      "antitrust violation",
      "governance failure",
      "shareholder lawsuit",
      "market manipulation"
    ]
  },
  "note": "Environment and Governance dictionary content reflects the visible portions of the supplied pasted files. Those files ended mid-dictionary, so these terms must be reviewed and completed before scoring production companies. Social terms reproduce the list supplied in the request and are grouped by theme for search only."
}

PILLAR_KEYWORDS = {
    "E": TAXONOMY["environment_keywords"],
    "S": TAXONOMY["social_keywords"],
    "G": TAXONOMY["governance_keywords"],
}
PILLAR_NAMES = {"E": "Environment", "S": "Social", "G": "Governance"}
print(TAXONOMY["note"])
print({pillar: len(topics) for pillar, topics in PILLAR_KEYWORDS.items()})


## 3. Extract report text and locate topic evidence

The extractor reads searchable text first and tries OCR on pages with little or no text. If Tesseract is missing or OCR fails, the page is flagged. Review the cited page and surrounding text before accepting any metric.

In [ ]:
def extract_pdf_pages(pdf_bytes):
    pages = []
    document = pymupdf.open(stream=pdf_bytes, filetype="pdf")
    for page_number, page in enumerate(document, start=1):
        text = page.get_text("text") or ""
        ocr_used = False
        extraction_issue = ""
        if len(re.sub(r"\s+", "", text)) < 60:
            try:
                text_page = page.get_textpage_ocr(language="eng", dpi=220, full=True)
                text = page.get_text("text", textpage=text_page) or text
                ocr_used = True
            except Exception as exc:
                extraction_issue = f"OCR unavailable or failed: {type(exc).__name__}: {exc}"
        pages.append({
            "page": page_number,
            "text": text,
            "characters": len(text),
            "ocr_used": ocr_used,
            "extraction_issue": extraction_issue,
        })
    if not any(p["text"].strip() for p in pages):
        raise ValueError("No usable text was extracted. Check the PDF and install Tesseract for scanned pages.")
    return pages

pages_by_ticker = {}
extraction_rows = []
for company in company_inputs:
    pages = extract_pdf_pages(company["content"])
    pages_by_ticker[company["ticker"]] = pages
    for page in pages:
        extraction_rows.append({"ticker": company["ticker"], "report_name": company["name"], **{k: v for k, v in page.items() if k != "text"}})
extraction_summary = pd.DataFrame(extraction_rows)
display(extraction_summary.head(50))
print(f"Processed {len(company_inputs)} reports and {len(extraction_summary):,} pages.")


In [ ]:
def compile_topic_patterns(pillar_keywords):
    compiled = {}
    for pillar, topics in pillar_keywords.items():
        compiled[pillar] = {}
        for topic, terms in topics.items():
            escaped = sorted({re.escape(t.strip()) for t in terms if t.strip()}, key=len, reverse=True)
            if escaped:
                compiled[pillar][topic] = re.compile(r"(?<!\w)(?:" + "|".join(escaped) + r")(?!\w)", flags=re.IGNORECASE)
    return compiled

TOPIC_PATTERNS = compile_topic_patterns(PILLAR_KEYWORDS)
NUMBER_PATTERN = re.compile(r"(?<!\w)(?:\d[\d,]*(?:\.\d+)?%?|\d+(?:\.\d+)?\s*(?:tonnes?|tons?|tCO2e|MWh|GWh|kWh|litres?|kilolitres?|employees?|beneficiaries))(?!\w)", re.I)
YEAR_PATTERN = re.compile(r"\b(?:19|20)\d{2}(?:\s*[-/–]\s*(?:19|20)?\d{2})?\b")
PROGRESS_PATTERN = re.compile(r"\b(target|goal|reduced|reduction|improved|increased|decreased|year[- ]on[- ]year|baseline|progress|achieved|aims to|committed to)\b", re.I)
TRACE_PATTERN = re.compile(r"\b(scope|baseline|intensity|per employee|per unit|per tonne|assured|assurance|verified|audited|boundary|coverage)\b", re.I)

def collect_topic_evidence(page_records, company):
    records = []
    for pillar, topic_patterns in TOPIC_PATTERNS.items():
        for topic, pattern in topic_patterns.items():
            for page in page_records:
                text = page["text"]
                matches = list(pattern.finditer(text))
                if not matches:
                    continue
                contexts = []
                for match in matches[:25]:
                    start = max(0, match.start() - 180)
                    end = min(len(text), match.end() + 260)
                    contexts.append(text[start:end].replace("\n", " "))
                joined = " ".join(contexts)
                records.append({
                    "ticker": company["ticker"], "company_name": company["company_name"], "report_name": company["name"], "pillar": pillar, "pillar_name": PILLAR_NAMES[pillar],
                    "topic": topic, "page": page["page"], "keyword_hits": len(matches),
                    "numeric_signal": bool(NUMBER_PATTERN.search(joined)),
                    "period_signal": bool(YEAR_PATTERN.search(joined)),
                    "progress_signal": bool(PROGRESS_PATTERN.search(joined)),
                    "traceability_signal": bool(TRACE_PATTERN.search(joined)),
                    "evidence_excerpt": contexts[0][:420],
                })
    return pd.DataFrame(records)

evidence_frames = [collect_topic_evidence(pages_by_ticker[c["ticker"]], c) for c in company_inputs]
evidence = pd.concat(evidence_frames, ignore_index=True) if evidence_frames else pd.DataFrame()
if evidence.empty:
    display(Markdown("No taxonomy terms were found in the uploaded reports. Check the PDF extraction and company reports."))
else:
    display(evidence.sort_values(["ticker", "pillar", "topic", "page"]).head(150))


## 4. Create the provisional document evidence score

The score measures whether the report contains topic references, quantitative signals, progress language and traceable evidence. It does **not** decide whether the reported outcome is good or bad. Equal topic weights are a prototype default until an industry materiality profile is reviewed.

In [ ]:
def calculate_document_scores(evidence, pillar_keywords, companies):
    rows = []
    for company in companies:
        for pillar, topic_map in pillar_keywords.items():
            topic_names = list(topic_map)
            p = evidence[(evidence["ticker"] == company["ticker"]) & (evidence["pillar"] == pillar)] if not evidence.empty else pd.DataFrame()
            by_topic = p.groupby("topic").agg(
                keyword_hits=("keyword_hits", "sum"),
                numeric_signal=("numeric_signal", "max"),
                period_signal=("period_signal", "max"),
                progress_signal=("progress_signal", "max"),
                traceability_signal=("traceability_signal", "max"),
                pages=("page", lambda x: sorted(set(x))),
            ) if not p.empty else pd.DataFrame()
            topic_count = max(len(topic_names), 1)
            present = set(by_topic.index) if not by_topic.empty else set()
            numeric = set(by_topic.index[by_topic["numeric_signal"]]) if not by_topic.empty else set()
            progress = set(by_topic.index[by_topic["progress_signal"]]) if not by_topic.empty else set()
            traceable = set(by_topic.index[by_topic["traceability_signal"] & by_topic["numeric_signal"]]) if not by_topic.empty else set()
            outcome_points = 4.0 * len(numeric) / topic_count
            management_points = 2.0 * len(progress) / topic_count
            evidence_points = 1.0 * len(traceable) / max(len(numeric), 1) if numeric else 0.0
            base = min(7.0, outcome_points + management_points + evidence_points)
            rows.append({
                "ticker": company["ticker"], "company_name": company["company_name"] or company["ticker"],
                "report_name": company["name"], "pillar": pillar, "pillar_name": PILLAR_NAMES[pillar],
                "document_evidence_score_0_7": round(base, 2),
                "topics_found": len(present), "topics_in_dictionary": topic_count,
                "topic_coverage_pct": round(100 * len(present) / topic_count, 1),
                "quantified_topics": len(numeric), "progress_topics": len(progress),
                "traceable_topics": len(traceable),
                "score_status": "PROVISIONAL: evidence coverage only",
            })
    return pd.DataFrame(rows)

document_scores = calculate_document_scores(evidence, PILLAR_KEYWORDS, company_inputs)
display(document_scores)

## 5. Find adverse news candidates with GDELT and Google News RSS

For each ticker, the notebook searches pillar-specific adverse-event terms in both sources. Google News RSS is an unofficial, best-effort feed surface, so its format, result count and history can change. FinBERT flags English-language candidates for analyst review. Negative sentiment does not establish that an ESG incident occurred.


In [ ]:
def canonical_url(url):
    parsed = urlparse(url)
    clean_query = [(k, v) for k, v in parse_qsl(parsed.query) if not k.lower().startswith(("utm_", "fbclid", "gclid"))]
    return urlunparse((parsed.scheme.lower(), parsed.netloc.lower().removeprefix("www."), parsed.path.rstrip("/"), "", urlencode(clean_query), ""))

def deduplicate_news(frame):
    if frame.empty:
        return frame
    frame = frame.drop_duplicates(subset=["ticker", "pillar", "canonical_url"]).reset_index(drop=True)
    keep = []
    for idx, row in frame.sort_values("published_at", ascending=False).iterrows():
        duplicate = any(
            row["ticker"] == frame.loc[prior, "ticker"]
            and row["pillar"] == frame.loc[prior, "pillar"]
            and SequenceMatcher(None, row["title"].lower(), frame.loc[prior, "title"].lower()).ratio() >= 0.94
            for prior in keep
        )
        if not duplicate:
            keep.append(idx)
    return frame.loc[keep].reset_index(drop=True)

def gdelt_query(entity, terms, start_date, end_date):
    selected = terms[:18]
    term_block = " OR ".join(f'"{term}"' if " " in term else term for term in selected)
    return {
        "query": f'"{entity}" ({term_block})', "mode": "artlist", "format": "json",
        "maxrecords": GDELT_MAX_RECORDS,
        "startdatetime": start_date.strftime("%Y%m%d000000"),
        "enddatetime": end_date.strftime("%Y%m%d235959"), "sort": "datedesc",
    }

def fetch_gdelt_company(company, scan_end):
    requested_start = scan_end - timedelta(days=NEWS_LOOKBACK_DAYS)
    source_start = max(requested_start, scan_end - timedelta(days=NEWS_SOURCE_WINDOW_DAYS - 1))
    entities = list(dict.fromkeys(([company["company_name"]] if company["company_name"] else [company["ticker"]]) + company["aliases"]))
    articles, errors = [], []
    session = requests.Session()
    session.headers.update({"User-Agent": "ESGResearchNotebook/0.2 research"})
    for pillar, terms in TAXONOMY["news_event_terms"].items():
        for entity in entities:
            params = gdelt_query(entity, terms, source_start, scan_end)
            try:
                response = session.get("https://api.gdeltproject.org/api/v2/doc/doc", params=params, timeout=30)
                response.raise_for_status()
                for item in response.json().get("articles", []):
                    url, title = item.get("url"), item.get("title", "").strip()
                    if not url or not title:
                        continue
                    articles.append({
                        "ticker": company["ticker"], "company_name": company["company_name"],
                        "pillar": pillar, "query_entity": entity, "title": title,
                        "url": url, "canonical_url": canonical_url(url),
                        "domain": item.get("domain", urlparse(url).netloc),
                        "published_at": item.get("seendate", ""), "language": item.get("language", ""),
                        "source_country": item.get("sourcecountry", ""), "discovery_source": "GDELT",
                    })
            except Exception as exc:
                errors.append({"ticker": company["ticker"], "source": "GDELT", "pillar": pillar, "query": entity, "error": str(exc)[:300]})
            time.sleep(0.2)
    coverage = {"ticker": company["ticker"], "source": "GDELT", "requested_start": requested_start.isoformat(),
                "source_start": source_start.isoformat(), "source_end": scan_end.isoformat(),
                "coverage_complete": source_start <= requested_start,
                "coverage_note": "Recent article-list window only; older requested dates are not covered" if source_start > requested_start else "Requested interval queried; gaps still apply"}
    return articles, errors, coverage

def google_news_feed_url(query):
    return "https://news.google.com/rss/search?" + urlencode({"q": query, "hl": "en-IN", "gl": "IN", "ceid": "IN:en"})

def fetch_google_news_company(company, scan_end):
    requested_start = scan_end - timedelta(days=NEWS_LOOKBACK_DAYS)
    source_start = max(requested_start, scan_end - timedelta(days=NEWS_SOURCE_WINDOW_DAYS - 1))
    entities = list(dict.fromkeys(([company["company_name"]] if company["company_name"] else [company["ticker"]]) + company["aliases"]))
    articles, errors = [], []
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; ESGResearchNotebook/0.2)"})
    for pillar, terms in TAXONOMY["news_event_terms"].items():
        selected = terms[:12]
        term_block = " OR ".join(f'"{term}"' if " " in term else term for term in selected)
        for entity in entities:
            query = f'"{entity}" ({term_block}) after:{source_start.isoformat()} before:{(scan_end + timedelta(days=1)).isoformat()}'
            try:
                response = session.get(google_news_feed_url(query), timeout=30)
                response.raise_for_status()
                root = ET.fromstring(response.content)
                items = root.findall(".//item")
                for item in items:
                    title = item.findtext("title", default="").strip()
                    url = item.findtext("link", default="").strip()
                    if not title or not url:
                        continue
                    source_node = item.find("source")
                    source_name = source_node.text.strip() if source_node is not None and source_node.text else ""
                    try:
                        published = parsedate_to_datetime(item.findtext("pubDate", default="")).isoformat()
                    except Exception:
                        published = item.findtext("pubDate", default="")
                    articles.append({
                        "ticker": company["ticker"], "company_name": company["company_name"],
                        "pillar": pillar, "query_entity": entity, "title": title,
                        "url": url, "canonical_url": canonical_url(url),
                        "domain": source_name or urlparse(url).netloc,
                        "published_at": published, "language": "", "source_country": "India",
                        "discovery_source": "Google News RSS",
                    })
            except Exception as exc:
                errors.append({"ticker": company["ticker"], "source": "Google News RSS", "pillar": pillar, "query": entity, "error": str(exc)[:300]})
            time.sleep(0.2)
    coverage = {"ticker": company["ticker"], "source": "Google News RSS", "requested_start": requested_start.isoformat(),
                "source_start": source_start.isoformat(), "source_end": scan_end.isoformat(),
                "coverage_complete": False,
                "coverage_note": "Best-effort RSS results; result cap, feed format and historical coverage are not guaranteed"}
    return articles, errors, coverage

news_articles, news_errors, coverage_rows = [], [], []
news_scan_date = date.today()
for company in company_inputs:
    for fetcher in (fetch_gdelt_company, fetch_google_news_company):
        rows, errors, coverage = fetcher(company, news_scan_date)
        news_articles.extend(rows)
        news_errors.extend(errors)
        coverage_rows.append(coverage)
news_candidates = deduplicate_news(pd.DataFrame(news_articles)) if news_articles else pd.DataFrame()
news_query_errors = pd.DataFrame(news_errors)
news_coverage = pd.DataFrame(coverage_rows)
print(f"News scan cutoff: {news_scan_date}. Found {len(news_candidates)} deduplicated candidates across {len(company_inputs)} companies.")
display(news_coverage)
if not news_query_errors.empty:
    display(news_query_errors.head(30))


In [ ]:
def fetch_article_text(url):
    try:
        import trafilatura
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return "", "publisher page unavailable"
        text = trafilatura.extract(downloaded, include_comments=False, include_tables=False, favor_recall=True) or ""
        return text[:12000], "" if len(text) >= 200 else "limited article text"
    except Exception as exc:
        return "", f"fetch failed: {type(exc).__name__}"

def load_finbert():
    from transformers import pipeline
    return pipeline("text-classification", model=MODEL_ID, tokenizer=MODEL_ID, device=-1)

def classify_news_with_finbert(candidates):
    if candidates.empty:
        return candidates.copy()
    result = candidates.copy()
    result["article_text"] = ""
    result["fetch_status"] = ""
    for ticker_value, group in result.groupby("ticker"):
        fetch_indices = list(group.index[:MAX_ARTICLE_FETCHES_PER_COMPANY])
        for idx in fetch_indices:
            text, status = fetch_article_text(result.loc[idx, "url"])
            result.loc[idx, "article_text"] = text
            result.loc[idx, "fetch_status"] = status
            time.sleep(0.15)
        result.loc[~result.index.isin(fetch_indices) & (result["ticker"] == ticker_value), "fetch_status"] = "not fetched: per-company prototype cap"
    result["model_input_source"] = result["article_text"].map(lambda text: "article text" if text else "headline only")
    result["finbert_label"] = "not scored: language not identified as English"
    result["finbert_confidence"] = float("nan")
    result["finbert_negative_probability"] = float("nan")
    result["needs_analyst_review"] = True
    from langdetect import detect, LangDetectException
    for idx, row in result.iterrows():
        input_text = str(row["article_text"] or row["title"])
        try:
            language = str(row.get("language", "")).lower()
            if language not in {"english", "en"}:
                language = detect(input_text[:1000])
        except LangDetectException:
            language = "unknown"
        result.loc[idx, "detected_language"] = language
    english = result["detected_language"].fillna("").str.lower().isin(["english", "en"])
    score_indices = list(result.index[english])
    if score_indices:
        classifier = load_finbert()
        model_texts = [str(result.loc[idx, "article_text"] or result.loc[idx, "title"])[:4000] for idx in score_indices]
        predictions = classifier(model_texts, top_k=None, truncation=True, max_length=512, batch_size=8)
        for idx, prediction in zip(score_indices, predictions):
            scores = {item["label"].lower(): float(item["score"]) for item in prediction}
            label = max(scores, key=scores.get)
            result.loc[idx, "finbert_label"] = label
            result.loc[idx, "finbert_confidence"] = scores[label]
            result.loc[idx, "finbert_negative_probability"] = scores.get("negative", 0.0)
            result.loc[idx, "needs_analyst_review"] = scores.get("negative", 0.0) >= NEGATIVE_PROBABILITY_REVIEW_FLAG
    return result

news_scored = classify_news_with_finbert(news_candidates) if not news_candidates.empty else pd.DataFrame()
if news_scored.empty:
    display(Markdown("No candidate articles were returned. Review the query coverage and company mappings."))
else:
    display(news_scored[["ticker", "pillar", "title", "domain", "published_at", "discovery_source", "detected_language", "model_input_source", "finbert_label", "finbert_negative_probability", "needs_analyst_review", "url"]].sort_values(["ticker", "finbert_negative_probability"], ascending=[True, False], na_position="last").head(100))


## 6. Review candidate incidents

FinBERT labels sentiment only. Review article text, source quality, company identity, event date, allegation status, company response, duplicates, and whether the event has already affected a reported KPI. Add only confirmed incidents to the table below. Require five independent sources for corroboration, or a final regulator finding, court judgment or official sanction. Count one underlying event once, not each article. Do not enter an allegation as a finding.

In [ ]:
# Add confirmed incidents after reviewing the candidate list. One row represents one underlying event.
# source_count is independent sources for this one event. Require at least five, unless status is a final regulator finding, final court judgment or official sanction.
# severity points: 0.5 limited verified harm, 1.0 material verified harm, 2.0 severe or persistent harm.
# Example row: {"ticker": "ABC.NS", "incident_id": "INC-001", "pillar": "E", "severity_points": 1.0, "status": "final regulator finding", "source_count": 3, "summary": "Verified discharge violation", "already_in_metric": False}
reviewed_incidents = pd.DataFrame(columns=[
    "ticker", "incident_id", "pillar", "severity_points", "status",
    "source_count", "summary", "already_in_metric"
])

# Example of an empty, reviewed incident register. Populate only after analyst review.
reviewed_incidents


## 7. Calculate the provisional scorecard

The output remains provisional because the current automated base score measures evidence coverage, and incident adjustments require analyst confirmation. A production ESG performance rating needs industry-specific metric weights, outcome thresholds and a wider validation set.

In [ ]:
def apply_reviewed_incidents(document_scores, incidents, news):
    out = document_scores.copy()
    penalties = {(ticker_value, pillar): 0.0 for ticker_value in document_scores["ticker"].unique() for pillar in PILLAR_NAMES}
    if not incidents.empty:
        required = {"ticker", "incident_id", "pillar", "severity_points", "status", "source_count", "summary", "already_in_metric"}
        missing = required - set(incidents.columns)
        if missing:
            raise ValueError(f"Incident table is missing fields: {sorted(missing)}")
        for _, item in incidents.iterrows():
            key = (str(item["ticker"]).upper(), item["pillar"])
            if key not in penalties:
                raise ValueError(f"Invalid ticker or pillar for incident {item['incident_id']}: {key}")
            if bool(item["already_in_metric"]):
                continue
            status = str(item["status"]).strip().lower()
            authoritative_statuses = {"final regulator finding", "final court judgment", "official sanction"}
            independent_sources = int(item["source_count"])
            if independent_sources < 5 and status not in authoritative_statuses:
                raise ValueError(
                    f"Incident {item['incident_id']} needs five independent sources or an accepted final official finding."
                )
            severity = float(item["severity_points"])
            if severity not in {0.5, 1.0, 2.0}:
                raise ValueError("Use an approved incident severity of 0.5, 1.0 or 2.0 points.")
            penalties[key] += severity
    out["approved_incident_penalty"] = [penalties[(row.ticker, row.pillar)] for row in out.itertuples()]
    out["provisional_score_0_7"] = (out["document_evidence_score_0_7"] - out["approved_incident_penalty"]).clip(lower=0).round(2)
    out["score_interpretation"] = "Automated evidence coverage less analyst-approved incident adjustments; not verified ESG performance"
    if not news.empty:
        negative = news[news["finbert_negative_probability"] >= NEGATIVE_PROBABILITY_REVIEW_FLAG].groupby("ticker").size()
        out["negative_news_candidates"] = out["ticker"].map(negative).fillna(0).astype(int)
    else:
        out["negative_news_candidates"] = 0
    overall = out.groupby(["ticker", "company_name"], as_index=False).agg(
        overall_score_0_21=("provisional_score_0_7", "sum"),
        negative_news_candidates=("negative_news_candidates", "max"),
    )
    overall["overall_score_0_21"] = overall["overall_score_0_21"].round(2)
    overall["esg_rank"] = overall["overall_score_0_21"].rank(method="min", ascending=False).astype(int)
    out = out.merge(overall[["ticker", "overall_score_0_21", "esg_rank"]], on="ticker", how="left")
    out["pillar_rank"] = out.groupby("pillar")["provisional_score_0_7"].rank(method="min", ascending=False).astype(int)
    return out.sort_values(["esg_rank", "pillar"], ascending=[True, True]).reset_index(drop=True)

scorecard = apply_reviewed_incidents(document_scores, reviewed_incidents, news_scored)
company_rankings = scorecard.pivot_table(
    index=["esg_rank", "ticker", "company_name", "overall_score_0_21", "negative_news_candidates"],
    columns="pillar", values="provisional_score_0_7", aggfunc="first",
).reset_index()
company_rankings.columns.name = None
company_rankings = company_rankings.rename(columns={"E": "E_score_0_7", "S": "S_score_0_7", "G": "G_score_0_7"})
print("Company ranking, highest provisional score first")
display(company_rankings.sort_values(["esg_rank", "ticker"]))
print("Pillar detail")
display(scorecard.sort_values(["esg_rank", "pillar"]))


## 8. Export the evidence pack

In [ ]:
run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
score_path = OUTPUT_DIR / f"esg_company_ranking_{run_stamp}.csv"
pillar_score_path = OUTPUT_DIR / f"esg_pillar_scores_{run_stamp}.csv"
evidence_path = OUTPUT_DIR / f"esg_report_evidence_{run_stamp}.csv"
news_path = OUTPUT_DIR / f"esg_news_candidates_{run_stamp}.csv"
query_path = OUTPUT_DIR / f"esg_news_query_errors_{run_stamp}.csv"
coverage_path = OUTPUT_DIR / f"esg_news_coverage_{run_stamp}.csv"

company_rankings.to_csv(score_path, index=False)
scorecard.to_csv(pillar_score_path, index=False)
evidence.to_csv(evidence_path, index=False)
news_scored.to_csv(news_path, index=False)
news_query_errors.to_csv(query_path, index=False)
news_coverage.to_csv(coverage_path, index=False)
print({"ranking": str(score_path), "pillar_scores": str(pillar_score_path), "report_evidence": str(evidence_path), "news_candidates": str(news_path), "news_query_errors": str(query_path), "news_coverage": str(coverage_path)})


## Interpretation and limits

- The ranking orders companies by provisional E, S and G report-evidence scores after analyst-approved incident adjustments. It is not a verified ESG performance rating.
- Negative FinBERT candidates appear beside each company score. They do not reduce the score until an analyst confirms and records the underlying incident.
- Google News RSS is an unofficial, best-effort discovery feed. Its result coverage, history and format may change. GDELT and Google News can miss relevant reporting or return duplicates.
- The score applies the same topic weights to every company. Industry materiality and metric thresholds are needed for defensible cross-sector comparisons.
- OCR, PDF tables, company aliases, publisher article access, language detection and incident grouping can fail. Review the evidence passages and sources.
- The Environmental and Governance pasted attachments were truncated. Complete and review those dictionaries before relying on comparisons.
- This notebook is an analyst workflow prototype. Validate it against manually reviewed reports and news before using rankings for investment decisions.
